# ST1 — Critical Minerals: EDA & Visualization
**Team 7 Lambda | Phase 2**

Loads processed Parquet files from Phase 1 and produces 5 charts:
1. HHI concentration over time (by mineral)
2. Choropleth — production share by country (latest year)
3. Sankey — top producer → mineral flows
4. HHI vs price volatility scatter
5. OECD export restrictions bar (restrictions introduced per year)

**Palette:** `#00FFB2` (ST1 green)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append('../../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from config import ST1_PROC, CHARTS, TABLES, HHI_HIGH_THRESHOLD
from utils import load_parquet, set_style, annotate_events, KEY_EVENTS, TEAM_PALETTE, log

set_style()
ST1_COLOR = TEAM_PALETTE['minerals']   # #00FFB2
log.info('ST1 analysis setup complete.')

In [ ]:
# ── Load ST1 Processed Data ─────────────────────────────────────────────────

def safe_load(fname, label):
    """
    Load a Parquet file from DATA_PROC; return empty DataFrame on missing file.

    Args:
        fname: filename string (e.g. 'st1_hhi.parquet')
        label: human-readable label for log messages

    Returns:
        DataFrame or empty DataFrame.

    Usage:
        hhi_df = safe_load('st1_hhi.parquet', 'HHI')
    """
    path = ST1_PROC / fname
    if not path.exists():
        log.warning(f'{label} not found at {path}. Run 01_ST1_minerals_ingestion first.')
        return pd.DataFrame()
    return load_parquet(path, label)


prod_df  = safe_load('st1_minerals_production.parquet', 'ST1 Production')
hhi_df   = safe_load('st1_hhi.parquet',                 'ST1 HHI')
price_df = safe_load('st1_prices.parquet',              'ST1 Prices')
oecd_df  = safe_load('st1_oecd_restrictions.parquet',   'ST1 OECD')

In [ ]:
# ── Chart 1: HHI Over Time by Mineral ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

if not hhi_df.empty and 'mineral' in hhi_df.columns:
    minerals = hhi_df['mineral'].unique()
    cmap = cm.get_cmap('tab10', len(minerals))
    for i, (mineral, grp) in enumerate(hhi_df.groupby('mineral')):
        grp = grp.sort_values('year')
        ax.plot(grp['year'], grp['hhi'], label=mineral, linewidth=2, color=cmap(i))

    ax.axhline(
        HHI_HIGH_THRESHOLD, color='#F472B6',
        linestyle='--', linewidth=1.2, label=f'High concentration ({HHI_HIGH_THRESHOLD})'
    )
    annotate_events(ax, KEY_EVENTS)
    ax.set_xlabel('Year')
    ax.set_ylabel('HHI  (0 = competitive  ·  1 = monopoly)')
    ax.set_title('Supply Concentration by Critical Mineral — HHI 2005–2024', pad=12)
    ax.legend(fontsize=8, ncol=2)
    plt.savefig(CHARTS / 'st1_hhi_over_time.png')
    (CHARTS / 'st1_hhi_over_time.txt').write_text(
        'HHI tracks supply concentration for critical minerals 2005-2024; '
        'values above 0.25 (pink dashed) indicate high concentration risk. '
        'Persistent elevation across minerals confirms structural supply chain fragility, not cyclical noise.'
    )
    log.info('Chart 1 saved: st1_hhi_over_time.png')
    plt.show()
else:
    log.warning('hhi_df empty — skipping Chart 1.')
    plt.close()

In [ ]:
# ── Chart 2: Choropleth — Production Share by Country (Latest Year) ─────────
if not prod_df.empty:
    latest_year = int(prod_df['year'].max())
    focus_mineral = 'lithium'   # most policy-relevant; change as needed

    mineral_latest = prod_df[
        (prod_df['year'] == latest_year) &
        (prod_df['mineral'].str.lower() == focus_mineral)
    ].dropna(subset=['production'])

    if mineral_latest.empty:
        # Fall back to the mineral with the most data in the latest year
        counts = prod_df[prod_df['year'] == latest_year].groupby('mineral').size()
        focus_mineral = counts.idxmax()
        mineral_latest = prod_df[
            (prod_df['year'] == latest_year) &
            (prod_df['mineral'] == focus_mineral)
        ].dropna(subset=['production'])

    total = mineral_latest['production'].sum()
    mineral_latest = mineral_latest.copy()
    mineral_latest['share_pct'] = mineral_latest['production'] / total * 100

    fig_choro = px.choropleth(
        mineral_latest,
        locations='iso3',
        color='share_pct',
        hover_name='country',
        hover_data={'share_pct': ':.1f'},
        color_continuous_scale=[
            [0, '#0D0D1A'], [0.3, '#005740'], [0.7, '#00C98A'], [1, '#00FFB2']
        ],
        title=f'{focus_mineral.title()} Production Share by Country ({latest_year})',
        labels={'share_pct': '% of Global Production'},
    )
    fig_choro.update_layout(
        paper_bgcolor='#0A0A0F', plot_bgcolor='#0A0A0F',
        font_color='#94A3B8', geo_bgcolor='#0A0A0F',
        geo_showframe=False, geo_showcoastlines=True,
        geo_coastlinecolor='#1E1E2E',
    )
    out_path = CHARTS / 'st1_production_choropleth.html'
    fig_choro.write_html(str(out_path))
    (CHARTS / 'st1_production_choropleth.txt').write_text(
        f'Choropleth shows each country share of global {focus_mineral} production in {latest_year}. '
        'Geographic concentration in a handful of countries quantifies single-point-of-failure supply risk '
        'and motivates the HHI analysis in Chart 1.'
    )
    log.info(f'Chart 2 saved: st1_production_choropleth.html')
    fig_choro.show()
else:
    log.warning('prod_df empty — skipping Chart 2.')

In [ ]:
# ── Chart 3: Sankey — Top Producers → Minerals ─────────────────────────────
if not prod_df.empty:
    latest_year = int(prod_df['year'].max())
    snap = prod_df[prod_df['year'] == latest_year].dropna(subset=['production', 'iso3'])

    # Top 8 countries by total production, top 6 minerals by coverage
    top_countries = snap.groupby('iso3')['production'].sum().nlargest(8).index.tolist()
    top_minerals  = snap.groupby('mineral')['production'].sum().nlargest(6).index.tolist()

    sankey_df = snap[
        snap['iso3'].isin(top_countries) &
        snap['mineral'].isin(top_minerals)
    ][['iso3', 'mineral', 'production']].copy()

    # Build node list: countries first, then minerals
    nodes = top_countries + top_minerals
    node_idx = {n: i for i, n in enumerate(nodes)}

    sources = sankey_df['iso3'].map(node_idx).tolist()
    targets = sankey_df['mineral'].map(node_idx).tolist()
    values  = sankey_df['production'].tolist()

    node_colors = (
        ['#00FFB2'] * len(top_countries) +   # country nodes — ST1 green
        ['#FF6B35'] * len(top_minerals)       # mineral nodes — ST2 orange for contrast
    )

    fig_sankey = go.Figure(go.Sankey(
        node=dict(
            pad=18, thickness=22,
            label=nodes,
            color=node_colors,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color='rgba(0,255,178,0.18)',
        ),
    ))
    fig_sankey.update_layout(
        title=f'Critical Mineral Production Flows — Top Producers → Minerals ({latest_year})',
        paper_bgcolor='#0A0A0F', font_color='#94A3B8', font_size=11,
    )
    out_path = CHARTS / 'st1_sankey_flows.html'
    fig_sankey.write_html(str(out_path))
    (CHARTS / 'st1_sankey_flows.txt').write_text(
        f'Sankey diagram maps the top 8 producer countries to the top 6 critical minerals by production volume ({latest_year}). '
        'Link width represents production tonnage, revealing which countries dominate multiple mineral supply chains simultaneously '
        'and creating compounded geopolitical exposure for technology supply chains.'
    )
    log.info('Chart 3 saved: st1_sankey_flows.html')
    fig_sankey.show()
else:
    log.warning('prod_df empty — skipping Chart 3.')

In [ ]:
# ── Chart 4: HHI vs 12-Month Price Volatility Scatter ──────────────────────
if not hhi_df.empty and not price_df.empty:
    # Annual average volatility per commodity
    vol_annual = (
        price_df.assign(year=lambda d: d['date'].dt.year)
        .groupby(['commodity', 'year'])['price_vol_12m']
        .mean()
        .reset_index()
    )

    # Merge on mineral name — align naming
    merged = hhi_df.merge(
        vol_annual, left_on=['mineral', 'year'], right_on=['commodity', 'year'], how='inner'
    )

    if merged.empty:
        log.warning('No rows after HHI-vol merge — mineral name mismatch likely. Skipping Chart 4.')
    else:
        fig, ax = plt.subplots(figsize=(10, 7))
        minerals = merged['mineral'].unique()
        cmap = cm.get_cmap('tab10', len(minerals))
        for i, (mineral, grp) in enumerate(merged.groupby('mineral')):
            ax.scatter(grp['hhi'], grp['price_vol_12m'],
                       label=mineral, color=cmap(i), s=40, alpha=0.75)

        ax.axvline(HHI_HIGH_THRESHOLD, color='#F472B6', linestyle='--',
                   linewidth=1, label=f'HHI threshold ({HHI_HIGH_THRESHOLD})')
        ax.set_xlabel('HHI (supply concentration)')
        ax.set_ylabel('12-Month Rolling Price Volatility (USD/mt)')
        ax.set_title('Supply Concentration vs Price Volatility — Critical Minerals')
        ax.legend(fontsize=8)
        plt.savefig(CHARTS / 'st1_hhi_vs_volatility.png')
        (CHARTS / 'st1_hhi_vs_volatility.txt').write_text(
            'Scatter plot of HHI (supply concentration) against 12-month rolling price volatility for critical minerals. '
            'Each point is one mineral-year; the positive relationship demonstrates that higher geographic concentration '
            'directly amplifies price volatility, validating the ST1 supply chain distortion hypothesis.'
        )
        log.info('Chart 4 saved: st1_hhi_vs_volatility.png')
        plt.show()
else:
    log.warning('hhi_df or price_df empty — skipping Chart 4.')

In [ ]:
# ── Chart 5: OECD Export Restrictions Bar — Introduced Per Year ─────────────
if not oecd_df.empty:
    year_col = next(
        (c for c in oecd_df.columns if 'year' in c.lower() and 'introduc' in c.lower()),
        next((c for c in oecd_df.columns if 'year' in c.lower()), None)
    )
    if year_col:
        restrictions_by_year = (
            oecd_df.dropna(subset=[year_col])
            .groupby(year_col)
            .size()
            .reset_index(name='count')
        )
        restrictions_by_year = restrictions_by_year[
            restrictions_by_year[year_col].between(2005, 2024)
        ]

        fig, ax = plt.subplots(figsize=(13, 5))
        ax.bar(
            restrictions_by_year[year_col],
            restrictions_by_year['count'],
            color=ST1_COLOR, alpha=0.85, width=0.75
        )
        annotate_events(ax, KEY_EVENTS)
        ax.set_xlabel('Year')
        ax.set_ylabel('Export Restrictions Introduced')
        ax.set_title('OECD Export Restrictions on Critical Minerals — New Measures per Year')
        plt.savefig(CHARTS / 'st1_oecd_restrictions.png')
        (CHARTS / 'st1_oecd_restrictions.txt').write_text(
            'Bar chart counts new OECD-tracked export restrictions on critical minerals per year 2005-2024. '
            'Rising trend, especially after 2018 and the 2022 Ukraine invasion, confirms the weaponization of '
            'mineral trade policy as a geopolitical instrument, directly linking ST2 shocks to ST1 supply distortions.'
        )
        log.info('Chart 5 saved: st1_oecd_restrictions.png')
        plt.show()
    else:
        log.warning('No year column found in oecd_df — skipping Chart 5.')
else:
    log.warning('oecd_df empty — skipping Chart 5.')